In [8]:
import sys, glob
spark_lib = '/opt/spark/python/lib'
for z in glob.glob(spark_lib + '/*.zip'):
    sys.path.insert(0, z)
sys.path.insert(0, '/opt/spark/python')
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("HiveAccessFromJupyter") \
    .master("spark://spark-master:7077") \
    .config("spark.hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.warehouse.dir", "hdfs://namenode:8020/user/hive/warehouse") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.maxResultSize", "200m") \
    .config("spark.sql.catalogImplementation", "hive") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print(spark.conf.get("spark.hive.metastore.uris"))
spark.sql("SHOW DATABASES").show()
spark.sql("SHOW TABLES IN default").show()

thrift://hive-metastore:9083
+---------+
|namespace|
+---------+
|  default|
+---------+

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [10]:
import pandas as pd

# Tampilkan semua kolom (tidak dipotong)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)        # Tidak batasi lebar tampilan
pd.set_option('display.max_rows', 10)  

In [11]:
spark.read.parquet("hdfs://namenode:8020/user/hive/warehouse/home_credit_gold").createOrReplaceTempView("Gold_Layer")
gold = spark.sql("""
SELECT *
FROM Gold_Layer
LIMIT 5
""")

gold.toPandas()

,loanId,target,ageYears,gender,has_car,has_house,asset_profile,children_cnt,family_members,income_per_member,contract_type,credit_amt,annuity_amt,debt_to_income,payment_to_income,has_down_payment,income_type_group,occupation_risk_group,stability_score,commute_risk_profile,region_rate,is_dini_hari,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,days_phone_change,has_sold,has_active,all_closed,active_loan_count,overdueSum,is_high_utilization,has_credit_limit,ever_overdue,is_recent_update,is_microloan,has_credit_card,has_collateral_loan,delinquent_months,ever_delinquent,ever_status_5,is_worsening_trend,history_length,active_bureau_loans,closed_bureau_loans,sold_bureau_loans,has_refused_prev,refused_count,is_high_risk_reason,is_recent_decision,has_pos_dpd,is_pos_high_risk,pos_history_length,is_pos_long_history,inst_late_ratio,is_inst_medium_late,inst_avg_payment_diff,inst_underpaid_freq,is_inst_version_0,cc_avg_utilization,is_cc_high_utilization,cc_recent_utilization,is_cc_low_pay_ratio,has_cc_activity,is_cc_accruing_debt
0,100002,1.0,26,M,0,1,Only House,0,1.0,202500.0,Cash loans,406597.5,24700.5,2.007889,0.121978,0,Private,High Risk,0,Stabil,2,0,0.083037,0.262949,0.139376,-1134.0,0.0,1.0,0.0,20.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,27.0,1.0,0.0,0.0,47.0,20.0,90.0,0.0,0,0,0,0,0,0,18,0,0.000000,0,0.000000,Tidak Pernah,0,NaN,NaN,NaN,NaN,NaN,NaN
1,100003,0.0,46,F,0,0,None,0,2.0,135000.0,Cash loans,1293502.5,35698.5,4.790750,0.132217,0,Stable,Medium Risk,0,Stabil,1,0,0.311267,0.622246,0.535276,-828.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,1.0,3.0,0.0,0,0,0,0,0,0,59,1,0.000000,0,0.000000,Tidak Pernah,0,NaN,NaN,NaN,NaN,NaN,NaN
2,100004,0.0,52,M,1,1,Both,0,1.0,67500.0,Revolving loans,135000.0,6750.0,2.000000,0.100000,0,Private,High Risk,1,Stabil,2,0,0.499629,0.555912,0.729567,-815.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,2.0,0.0,0,0,0,0,0,0,3,0,0.000000,0,0.000000,Tidak Pernah,0,NaN,NaN,NaN,NaN,NaN,NaN
3,100006,0.0,52,F,0,1,Only House,0,2.0,67500.0,Cash loans,312682.5,29686.5,2.316167,0.219900,0,Private,High Risk,1,Stabil,2,0,0.499629,0.650442,0.535276,-617.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1,0,0,0,19,0,0.000000,0,0.000000,Tidak Pernah,0,0.0,0.0,0.0,0.0,0.0,0.0
4,100007,0.0,55,M,0,1,Only House,0,1.0,121500.0,Cash loans,513000.0,21865.5,4.222222,0.179963,0,Private,Medium Risk,1,Stabil,2,0,0.499629,0.322738,0.535276,-1106.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,1.0,0.0,0,0,0,0,0,0,76,1,0.242424,0,-452.384318,Jarang,0,NaN,NaN,NaN,NaN,NaN,NaN
